# Práctica 2 – Análise de datasets e aplicación de técnicas de IA en Python 
## (Segunda Parte: Adestramento dos modelos)
### Modelos do grupo: AdaBoost e GAN

- Laura Cabaleiro Pintos
- Pablo Chantada Saborido
- Marcelo Ferreiro Sánchez
- José Romero Conde


## Polo momento probei co AdaBoost porque é o mais sinxelo para comezar.
Fago o caso binario e o caso multiclase, probei a facelo flow Xerárquico (Seguir o clasificador binario dun clasficador só adestrado coas instancias de ataque) mais iba para o carallo porque o erro do primeiro clasificador baixaba o accuracy do conxunto a 0.2 (supoño que se balanceamos mais o dataset no caso binario para minimizar ó máximo o seu erro podería ir ben a idea). De momento o que hai é un gridsearch de parámetros en ambos casos e xa, pódese encontrar mellor os óptimos isto é só proba inicial.


Respecto ao GAN, creo que a única opción para usalo é adestralo só coas instancias benignas e así poder usalo como discriminador. A idea é que se lle presentamos un ataque diga "nin de coña eu fixen isto, é demasiado diferente das cousas que eu xenero". Nese caso, a parte anterior de balanceo do dataset foi unha merda porque o GAN sí que poderíamos intentar adestralo con todas as instancias BENIGN do dataset.

Outra cousa que podemos facer así de extra é adestrar outro GAN para que faga máis datos sintéticos, sobretodo entre as clases de ataque, xa que segue existindo desbalanceos de ata 25:1.



# Pipeline que creo que estaría ben sería:

## AdaBoost:
- Balancear o dataset no caso binario (50% benign)
- Facer unha boa búsqueda de óptimos (quizá Bayesian Search)
- Se podemos, balancear tamén o dataset no caso multiclase utilizando a GAN (ou algunha ferramenta doutra librería se é mellor)

## GAN:
- Adestrala só cos exemplos do dataset deste notebook, ver seus resultados.
- Adestrala con tantos casos positivos como podamos, comparar os resultados co caso anterior.
- O multiclase aquí debe ser complicado, non sei se habería que facer unha GAN por clase e sometelas a votación (é o que se me ocorre).

## Extra:
- Intentar adestrar unha tamén para Data Augmentation no AdaBoost (moi moi opcional).

Imports

In [4]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np


Cargar o dataset

In [10]:
# Cargar datos
train = pd.read_parquet("dataset/train.parquet")
val = pd.read_parquet("dataset/val.parquet")
test = pd.read_parquet("dataset/test.parquet")

# Separar features y labels
X_train = train.drop(columns=['Label'])
y_train = train['Label']
X_val = val.drop(columns=['Label'])
y_val = val['Label']
X_test = test.drop(columns=['Label'])
y_test = test['Label']

Facer labels para o caso binario

In [6]:
# Preparar versión binaria
y_train_bin = (y_train != 'BENIGN').astype(int)
y_val_bin = (y_val != 'BENIGN').astype(int)
y_test_bin = (y_test != 'BENIGN').astype(int)

## Caso Binario

In [7]:
print("="*60)
print("OPTIMIZACIÓN ADA BOOST BINARIO")
print("="*60)
print(f"Forma do conx. de adestramento: {X_train.shape}")
print(f"BENIGN={sum(y_train_bin==0)}, ATTACK={sum(y_train_bin==1)}\n")





param_grid_bin = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.1, 0.5, 1.0, 1.5]
}

grid_bin = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid_bin,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)


grid_bin.fit(X_train, y_train_bin)

print(f"\nMellores parámetros binario: {grid_bin.best_params_}")
print(f"Mellor F1-score (CV): {grid_bin.best_score_:.4f}")

# Entrena o mellor modelo binario
best_ada_bin = grid_bin.best_estimator_
y_pred_bin = best_ada_bin.predict(X_test)

print(f"\n=== RESULTADOS BINARIO (TEST) ===")
print(f"Accuracy: {accuracy_score(y_test_bin, y_pred_bin):.4f}")
print(f"F1-Score: {f1_score(y_test_bin, y_pred_bin):.4f}")
print("\nReporte:")
print(classification_report(y_test_bin, y_pred_bin, target_names=['BENIGN', 'ATTACK']))

OPTIMIZACIÓN ADA BOOST BINARIO
Forma do conx. de adestramento: (127400, 52)
BENIGN=35000, ATTACK=92400

Fitting 3 folds for each of 16 candidates, totalling 48 fits

Mellores parámetros binario: {'learning_rate': 1.5, 'n_estimators': 300}
Mellor F1-score (CV): 0.9251

=== RESULTADOS BINARIO (TEST) ===
Accuracy: 0.8872
F1-Score: 0.9265

Reporte:
              precision    recall  f1-score   support

      BENIGN       0.92      0.64      0.76      7500
      ATTACK       0.88      0.98      0.93     19800

    accuracy                           0.89     27300
   macro avg       0.90      0.81      0.84     27300
weighted avg       0.89      0.89      0.88     27300



## Caso Multiclase

In [8]:
param_grid_multi = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.5, 1.0, 1.5]
}

grid_multi = GridSearchCV(
    AdaBoostClassifier(random_state=42),
    param_grid_multi,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)


grid_multi.fit(X_train, y_train)

print(f"\nMellores parámetros multiclase: {grid_multi.best_params_}")
print(f"Mellor F1-weighted (CV): {grid_multi.best_score_:.4f}")

# Entrena o mellor modelo multiclase
best_ada_multi = grid_multi.best_estimator_
y_pred_multi = best_ada_multi.predict(X_test)

print(f"\n=== RESULTADOS MULTICLASE (TEST) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_multi):.4f}")
print("\nReporte (solo top 5 clases):")
print(classification_report(y_test, y_pred_multi))


Fitting 3 folds for each of 9 candidates, totalling 27 fits

Mellores parámetros multiclase: {'learning_rate': 1.5, 'n_estimators': 300}
Mellor F1-weighted (CV): 0.5074

=== RESULTADOS MULTICLASE (TEST) ===
Accuracy: 0.4966

Reporte (solo top 5 clases):
                            precision    recall  f1-score   support

                    BENIGN       0.37      0.46      0.41      7500
                       Bot       0.00      0.00      0.00       750
                      DDoS       0.45      0.99      0.62      3000
             DoS GoldenEye       0.77      0.60      0.67      1650
                  DoS Hulk       0.85      0.26      0.40      3000
          DoS Slowhttptest       0.17      0.03      0.05      1500
             DoS slowloris       0.56      0.02      0.04      1500
               FTP-Patator       0.99      0.50      0.66      1500
                Heartbleed       0.00      0.00      0.00       300
              Infiltration       0.53      0.11      0.18       3

/home/marce/cyber/cyber/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/marce/cyber/cyber/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/marce/cyber/cyber/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

## Comparación

In [9]:

print("COMPARACIÓN DE CONFIGURACIÓNS")


configs = [
    {"name": "1", "params": {"n_estimators": 100, "learning_rate": 1.0}},
    {"name": "2", "params": {"n_estimators": 300, "learning_rate": 0.1}},
    {"name": "3", "params": {"n_estimators": 50, "learning_rate": 1.5}},
    {"name": "4", "params": {"n_estimators": 200, "learning_rate": 0.5}},
]

results = []

print("\nProbando configuracións en BINARIO:")
for config in configs:
    ada = AdaBoostClassifier(**config['params'], random_state=42)
    ada.fit(X_train, y_train_bin)
    y_pred = ada.predict(X_test)
    acc = accuracy_score(y_test_bin, y_pred)
    f1 = f1_score(y_test_bin, y_pred)
    results.append({
        "Modelo": "Binario",
        "Config": config['name'],
        "n_estimators": config['params']['n_estimators'],
        "learning_rate": config['params']['learning_rate'],
        "Accuracy": acc,
        "F1": f1
    })
    print(f"  {config['name']} (n={config['params']['n_estimators']}, lr={config['params']['learning_rate']}): Acc={acc:.4f}, F1={f1:.4f}")

print("\nProbando configuracións en MULTICLASE:")
for config in configs:
    ada = AdaBoostClassifier(**config['params'], random_state=42)
    ada.fit(X_train, y_train)
    y_pred = ada.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results.append({
        "Modelo": "Multiclase",
        "Config": config['name'],
        "n_estimators": config['params']['n_estimators'],
        "learning_rate": config['params']['learning_rate'],
        "Accuracy": acc,
        "F1": f1
    })
    print(f"  {config['name']} (n={config['params']['n_estimators']}, lr={config['params']['learning_rate']}): Acc={acc:.4f}, F1={f1:.4f}")

# Mostrar mejores
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("MELLORES CONFIGURACIÓNS")
print("="*60)

for modelo in ['Binario', 'Multiclase']:
    best = results_df[results_df['Modelo'] == modelo].sort_values('F1', ascending=False).iloc[0]
    print(f"\n{modelo}:")
    print(f"  Config: {best['Config']}")
    print(f"  n_estimators: {int(best['n_estimators'])}")
    print(f"  learning_rate: {best['learning_rate']}")
    print(f"  Accuracy: {best['Accuracy']:.4f}")
    print(f"  F1-Score: {best['F1']:.4f}")



COMPARACIÓN DE CONFIGURACIÓNS

Probando configuracións en BINARIO:
  1 (n=100, lr=1.0): Acc=0.8651, F1=0.9148
  2 (n=300, lr=0.1): Acc=0.8452, F1=0.9036
  3 (n=50, lr=1.5): Acc=0.8778, F1=0.9211
  4 (n=200, lr=0.5): Acc=0.8458, F1=0.9039

Probando configuracións en MULTICLASE:
  1 (n=100, lr=1.0): Acc=0.4380, F1=0.3627
  2 (n=300, lr=0.1): Acc=0.3710, F1=0.2327
  3 (n=50, lr=1.5): Acc=0.3483, F1=0.2837
  4 (n=200, lr=0.5): Acc=0.4716, F1=0.3652

MELLORES CONFIGURACIÓNS

Binario:
  Config: 3
  n_estimators: 50
  learning_rate: 1.5
  Accuracy: 0.8778
  F1-Score: 0.9211

Multiclase:
  Config: 4
  n_estimators: 200
  learning_rate: 0.5
  Accuracy: 0.4716
  F1-Score: 0.3652
